# M7 Code Cookbook

Hands-on companion for `m7-concepts-reference.md`.

Each section below culminates in one small runnable Python recipe. The examples
use the same Chronos Wealth teaching story:

```text
Can Alice add more AAPL under the guideline?
```

The snippets are intentionally plain Python. Run them from the repository root
unless a snippet says otherwise.

## 1. Agent Loop Anatomy

This recipe demonstrates the basic loop:

```text
perception -> planning -> tool execution -> observation -> planning
```


In [ ]:
state = {
    "question": "Can Alice add more AAPL under the guideline?",
    "observations": [],
}

def planner(state: dict) -> dict:
    if not state["observations"]:
        return {"tool": "get_current_price", "args": {"symbol": "AAPL"}}
    return {"final": f"Checked facts: {state['observations']}"}

def get_current_price(symbol: str) -> dict:
    return {"symbol": symbol, "price": 108.0}

for turn in range(3):
    step = planner(state)
    print("PLAN:", step)

    if "final" in step:
        print("FINAL:", step["final"])
        break

    # Actual tool execution happens here.
    result = get_current_price(**step["args"])
    observation = {"tool": step["tool"], "result": result}
    state["observations"].append(observation)
    print("OBSERVE:", observation)


Expected shape of output:

```text
PLAN: {'tool': 'get_current_price', ...}
OBSERVE: {'tool': 'get_current_price', 'result': ...}
PLAN: {'final': 'Checked facts: ...'}
FINAL: Checked facts: ...
```

What this proves:

- the loop is ordinary Python control flow;
- the planner chooses the next step;
- Python executes the tool;
- observations make the next planning turn better.

## 2. Tool Use And Function Calling

This recipe demonstrates structured tool requests, schema validation, one local
tool execution, and the place where a local model planner can be injected.

It runs without model dependencies by using `debug_model_output()`. If
`transformers`, `torch`, and `OFFLINE-AI-Models/smollm2-135m-instruct` are
available, switch `USE_LOCAL_MODEL` to `True`.

Important distinction:

```text
The model outputs a JSON tool request.
Python parses, validates, and executes the actual function call.
```


In [ ]:
import json
from pathlib import Path
from pydantic import BaseModel, Field

USE_LOCAL_MODEL = False

class CurrentPriceArgs(BaseModel):
    symbol: str = Field(description="Ticker symbol, for example AAPL")

def get_current_price(symbol: str) -> dict:
    prices = {"AAPL": 108.0, "MSFT": 196.0}
    return {"symbol": symbol, "price": prices[symbol]}

def debug_model_output(prompt: str) -> str:
    return '{"tool":"get_current_price","args":{"symbol":"AAPL"}}'

def ask_local_model(prompt: str) -> str:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch

    model_path = Path("OFFLINE-AI-Models/smollm2-135m-instruct")
    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(model_path, local_files_only=True).eval()

    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=60, do_sample=False)
    return tokenizer.decode(output[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

def model_planner(messages: list[dict]) -> str:
    transcript = "\n".join(message["content"] for message in messages)
    prompt = (
        "Return JSON only for the next tool call.\n"
        'Available tool: {"tool":"get_current_price","args":{"symbol":"AAPL"}}\n'
        f"Transcript:\n{transcript}"
    )
    return ask_local_model(prompt) if USE_LOCAL_MODEL else debug_model_output(prompt)

messages = [{"role": "user", "content": "Can Alice add more AAPL?"}]
raw_step = model_planner(messages)
print("RAW MODEL OUTPUT:", raw_step)

# The model requested a tool. Python now owns execution.
step = json.loads(raw_step)
args = CurrentPriceArgs.model_validate(step["args"])
print("VALIDATED TOOL CALL:", step["tool"], args.model_dump())

# Actual tool execution happens here.
result = get_current_price(**args.model_dump())
observation = {"tool": step["tool"], "result": result}
print("OBSERVATION:", observation)


What this proves:

- model planning output is text;
- function calling is structured text;
- Pydantic validates model-proposed arguments;
- the model requests the tool call, but Python executes it;
- the tool result becomes an observation;
- the LLM injection point is `model_planner(messages)`.

## 3. Bare-Metal Runtime

This recipe demonstrates parse, validate, dispatch, observe, repeat, and error
feedback in one small runtime.


In [ ]:
import json
from pydantic import BaseModel

class CurrentPriceArgs(BaseModel):
    symbol: str

class PortfolioAllocationArgs(BaseModel):
    client_id: int

class GuidelineCheckArgs(BaseModel):
    symbol: str
    proposed_allocation_pct: float

def get_current_price(symbol: str) -> dict:
    prices = {"AAPL": 108.0, "MSFT": 196.0}
    if symbol not in prices:
        raise ValueError(f"Unknown symbol {symbol}")
    return {"symbol": symbol, "price": prices[symbol]}

def get_portfolio_allocation(client_id: int) -> dict:
    return {"client_id": client_id, "AAPL": 32.0, "cash": 18.0}

def check_guidelines(symbol: str, proposed_allocation_pct: float) -> dict:
    return {
        "symbol": symbol,
        "allowed": proposed_allocation_pct <= 35.0,
        "limit_pct": 35.0,
    }

TOOL_SCHEMAS = {
    "get_current_price": CurrentPriceArgs,
    "get_portfolio_allocation": PortfolioAllocationArgs,
    "check_guidelines": GuidelineCheckArgs,
}
TOOL_FUNCTIONS = {
    "get_current_price": get_current_price,
    "get_portfolio_allocation": get_portfolio_allocation,
    "check_guidelines": check_guidelines,
}

def execute_tool(raw_step: str) -> dict:
    step = json.loads(raw_step)
    name = step["tool"]
    if name not in TOOL_FUNCTIONS:
        raise ValueError(f"Unknown tool: {name}")
    args = TOOL_SCHEMAS[name].model_validate(step["args"])
    # Generic dispatch: Python calls the selected tool here.
    return TOOL_FUNCTIONS[name](**args.model_dump())

def safe_execute(raw_step: str) -> dict:
    try:
        return {"ok": True, "result": execute_tool(raw_step)}
    except Exception as error:
        return {
            "ok": False,
            "error_type": type(error).__name__,
            "message": str(error),
        }

def debug_planner(messages: list[dict]) -> str:
    transcript = "\n".join(message["content"] for message in messages)
    if "get_current_price observation" not in transcript:
        return '{"tool":"get_current_price","args":{"symbol":"AAPL"}}'
    if "get_portfolio_allocation observation" not in transcript:
        return '{"tool":"get_portfolio_allocation","args":{"client_id":1}}'
    if "check_guidelines observation" not in transcript:
        return '{"tool":"check_guidelines","args":{"symbol":"AAPL","proposed_allocation_pct":36}}'
    return '{"final":"Alice should not raise AAPL to 36%; the guideline limit is 35%."}'

messages = [{"role": "user", "content": "Can Alice add more AAPL under guidelines?"}]

for turn in range(5):
    raw_step = debug_planner(messages)
    step = json.loads(raw_step)

    if "final" in step:
        print("FINAL:", step["final"])
        break

    outcome = safe_execute(raw_step)
    print("TURN", turn, "OUTCOME:", outcome)

    messages.append({
        "role": "tool",
        "content": f"{step['tool']} observation: {outcome}",
    })


What this proves:

- the runtime parses model-shaped text;
- the runtime validates against known schemas;
- the runtime dispatches only registered tools;
- observations are fed back to the planner;
- errors can become structured feedback instead of crashes.

## 4. Telemetry Logging

This recipe adds trace records to the runtime so the run can be inspected after
the final answer.


In [ ]:
import json
from time import perf_counter
from pydantic import BaseModel

class GuidelineCheckArgs(BaseModel):
    symbol: str
    proposed_allocation_pct: float

def check_guidelines(symbol: str, proposed_allocation_pct: float) -> dict:
    return {
        "symbol": symbol,
        "allowed": proposed_allocation_pct <= 35.0,
        "limit_pct": 35.0,
    }

TOOL_SCHEMAS = {"check_guidelines": GuidelineCheckArgs}
TOOL_FUNCTIONS = {"check_guidelines": check_guidelines}
trace = []

def execute_with_trace(turn: int, raw_step: str) -> dict:
    started = perf_counter()
    record = {"turn": turn, "raw_model_output": raw_step}
    try:
        step = json.loads(raw_step)
        record["tool"] = step.get("tool")
        record["raw_args"] = step.get("args")

        args = TOOL_SCHEMAS[step["tool"]].model_validate(step["args"])
        record["validated_args"] = args.model_dump()

        # Traced dispatch: Python calls the selected tool here.
        result = TOOL_FUNCTIONS[step["tool"]](**args.model_dump())
        record["result"] = result
        return result
    except Exception as error:
        record["exception"] = type(error).__name__
        record["message"] = str(error)
        raise
    finally:
        record["elapsed_ms"] = round((perf_counter() - started) * 1000, 2)
        trace.append(record)

good_step = '{"tool":"check_guidelines","args":{"symbol":"AAPL","proposed_allocation_pct":36}}'
bad_step = '{"tool":"check_guidelines","args":{"ticker":"AAPL","proposed_allocation_pct":36}}'

for turn, raw_step in enumerate([good_step, bad_step]):
    try:
        print("RESULT:", execute_with_trace(turn, raw_step))
    except Exception:
        print("ERROR RECORDED")

print("\nTRACE")
for record in trace:
    print(json.dumps(record))


What this proves:

- trace records explain success and failure;
- raw model output is preserved;
- validated arguments are visible;
- exceptions become inspectable events;
- telemetry is part of the agent runtime, not decoration.

## Lab Culmination

After completing all four recipes, the full agent architecture is visible:

```text
messages/transcript      -> perception
debug_planner/model      -> planning
JSON tool request        -> proposed action
Pydantic schema          -> typed boundary
Python tool function     -> execution
observation append       -> feedback
trace record             -> audit/debug view
max turns                -> safety stop
```
